# ESM3 Multimodal Protein Language Model

![ESM3 Multimodal Protein Language Model](https://proto-bio.github.io/proto-assets/images/tool/esm3/hero.png)

This notebook demonstrates the three core operations of the ESM3 tool: extracting sequence embeddings, sampling mutations, and scoring sequences. Unlike sequence-only models, ESM3 was trained on multimodal protein data covering sequence, structure, and function, allowing it to capture richer relationships between amino acid identity, three-dimensional geometry, and biological annotation. Even when working with sequence inputs alone, ESM3's representations encode structural and functional context that purely sequence-based models miss.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm3")
display_overview("esm3")
display_docs_section("esm3", "Background")

## Available tools

In [ ]:
display_available_tools("esm3")

### `run_esm3_embeddings`

ESM3 produces dense vector representations for each input protein sequence. Because the model was trained on multimodal data, these embeddings encode not only evolutionary patterns but also structural and functional context. The mean-pooled vectors can be used for similarity search, clustering, classification, and as input features for downstream machine learning models.

In [ ]:
from proto_tools import (
    ESM3EmbeddingsConfig,
    ESM3EmbeddingsInput,
    run_esm3_embeddings,
)

In [ ]:
# Display docs
display_api_reference("esm3", "input", "run_esm3_embeddings")

# Two hemoglobin-like sequences to embed
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM3EmbeddingsInput(sequences=sequences)

In [ ]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_embeddings")

# Configure the open-source ESM3 model | see docs above for all fields
config = ESM3EmbeddingsConfig(
    model_checkpoint="esm3_sm_open_v1",
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the embeddings function
embeddings_result = run_esm3_embeddings(inputs, config)

In [ ]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_embeddings")

# Inspect the embedding dimension and first few values for each sequence
for i, emb in enumerate(embeddings_result.results):
    print(f"Sequence {i + 1}: embedding length = {len(emb.mean_embedding)}, first 5 values = {emb.mean_embedding[:5]}")

### `run_esm3_sample`

ESM3 sampling uses the model's predicted amino acid distributions to propose mutations at masked positions. Because ESM3 was trained on multimodal data, its position confidence estimates reflect structural and functional plausibility in addition to evolutionary conservation. There are two ways to specify which positions to sample:

1. **Custom masks** — directly mark positions with `_` in the input sequence to control exactly which residues are redesigned
2. **Masking strategy** — let the `MaskingStrategy` config automatically select positions using random, entropy-based, or max-logit methods

#### Approach 1: Custom masks with `_`

Use the `_` character to explicitly mark which positions should be redesigned. All other positions are held fixed. This is useful when you know exactly which residues you want to mutate — for example, redesigning a loop region or a specific binding interface.

In [ ]:
from proto_tools import (
    ESM3SampleConfig,
    ESM3SampleInput,
    run_esm3_sample,
)

# Display input docs
display_api_reference("esm3", "input", "run_esm3_sample")

# Mask positions 6-10 with _ to redesign that region
masked_input = ESM3SampleInput(sequences=["MVLSP_____VKAAW"])

In [ ]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_sample")

# No masking strategy needed — positions are already specified by _
config = ESM3SampleConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run sampling on the custom-masked input
custom_mask_result = run_esm3_sample(masked_input, config)

In [ ]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_sample")

# Show the original vs sampled — only the masked positions change
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(custom_mask_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

#### Approach 2: Automatic position selection with `MaskingStrategy`

Instead of manually placing `_` characters, you can let `MaskingStrategy` choose which positions to mask. The `method` parameter controls the selection logic:

- **`"random"`** (default) — uniform random selection
- **`"entropy"`** — targets positions where ESM3 is least certain about the original residue
- **`"max-logit"`** — targets positions where ESM3 most confidently predicts a different amino acid

Use `num_mutations` for an exact count or `mask_fraction` for a proportion of positions.

In [ ]:
from proto_tools.transforms.masking import MaskingStrategy

# Entropy-based masking to target the 5 most uncertain positions
strategy_input = ESM3SampleInput(sequences=["MVLSPADKTNVKAAW"])
strategy_config = ESM3SampleConfig(
    masking_strategy=MaskingStrategy(method="entropy", num_mutations=5),
    device="cuda",  # Change to "cpu" if no GPU available
)

strategy_result = run_esm3_sample(strategy_input, strategy_config)

In [ ]:
# The model automatically selected the 5 highest-entropy positions to mutate
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(strategy_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

### `run_esm3_score`

Pseudo-perplexity scoring masks each position in a sequence one at a time and measures how well the model predicts the original residue. Because ESM3 was trained on multimodal data, its perplexity scores reflect not just evolutionary conservation but also structural and functional plausibility. Lower perplexity indicates that the sequence is more consistent with the patterns the model learned during training. This is useful for ranking designed sequences, filtering out poor candidates before experimental validation, or identifying anomalous regions within a protein.

In [ ]:
from proto_tools import (
    ESM3ScoringConfig,
    ESM3ScoringInput,
    run_esm3_score,
)

In [ ]:
# Display docs
display_api_reference("esm3", "input", "run_esm3_score")

# Two short sequences to score
inputs = ESM3ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

In [ ]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_score")

# Process 16 masked variants per forward pass | see docs above for all fields
config = ESM3ScoringConfig(
    batch_size=16,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the scoring function
score_result = run_esm3_score(inputs, config)

In [ ]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_score")

# Show all scoring metrics for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {inputs.sequences[i]}")
    print(f"  Log-likelihood:     {score['log_likelihood']:.3f}")
    print(f"  Avg log-likelihood: {score['avg_log_likelihood']:.3f}")
    print(f"  Perplexity:         {score['perplexity']:.3f}")

## Export Results

Results from each function can be exported to standard file formats for downstream analysis. Embeddings export to CSV with one row per sequence, sampled sequences export to FASTA for compatibility with other bioinformatics tools, and scores export to JSON with full metadata.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export("esm3_embeddings", export_path=output_dir, file_format="csv")
print(f"Embeddings exported to {output_dir / 'esm3_embeddings.csv'}")

strategy_result.export("esm3_sequences", export_path=output_dir, file_format="fasta")
print(f"Sampled sequences exported to {output_dir / 'esm3_sequences.fasta'}")

score_result.export("esm3_scores", export_path=output_dir, file_format="json")
print(f"Scores exported to {output_dir / 'esm3_scores.json'}")